In [3]:
# =============================================
# TUGAS 4 - NO. 2: PEMBOBOTAN (TERM WEIGHTING)
# Nama: Abdul Rahman
# NIM : 240210502033
# =============================================

import re
import math
import pandas as pd
import numpy as np

# Install Sastrawi library if not already installed
!pip install Sastrawi

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer

# Setup Sastrawi
stemmer = StemmerFactory().create_stemmer()
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

print("Library berhasil di-import.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 6.8 MB/s eta 0:00:00
Library berhasil di-import.


In [4]:
# Dataset: 4 dokumen pendek berbahasa Indonesia (topik komputer & teknologi)
dokumen = [
    "Sistem komputer adalah perangkat yang terdiri dari hardware dan software.",
    "Jaringan komputer menghubungkan beberapa komputer untuk berbagi data.",
    "Kecerdasan buatan memungkinkan komputer belajar dari data.",
    "Sistem temu kembali informasi menggunakan komputer untuk mencari dokumen."
]

print("Dataset Dokumen:")
for i, doc in enumerate(dokumen, 1):
    print(f"Dok {i}: {doc}")

Dataset Dokumen:
Dok 1: Sistem komputer adalah perangkat yang terdiri dari hardware dan software.
Dok 2: Jaringan komputer menghubungkan beberapa komputer untuk berbagi data.
Dok 3: Kecerdasan buatan memungkinkan komputer belajar dari data.
Dok 4: Sistem temu kembali informasi menggunakan komputer untuk mencari dokumen.


In [5]:
def preprocess_sederhana(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = stopword_remover.remove(text)
    tokens = text.split()
    return tokens

tokens_dokumen = [preprocess_sederhana(doc) for doc in dokumen]
print("Hasil Preprocessing (Token):")
for i, tokens in enumerate(tokens_dokumen, 1):
    print(f"Dok {i}: {tokens}")

Hasil Preprocessing (Token):
Dok 1: ['sistem', 'komputer', 'perangkat', 'terdiri', 'hardware', 'software']
Dok 2: ['jaringan', 'komputer', 'menghubungkan', 'beberapa', 'komputer', 'berbagi', 'data']
Dok 3: ['kecerdasan', 'buatan', 'memungkinkan', 'komputer', 'belajar', 'data']
Dok 4: ['sistem', 'temu', 'informasi', 'menggunakan', 'komputer', 'mencari', 'dokumen']


In [6]:
# Bangun vocabulary (semua term unik dari seluruh dokumen)
vocab = sorted(set(term for tokens in tokens_dokumen for term in tokens))
print("Vocabulary:", vocab)
print("Jumlah term unik:", len(vocab))

# Bangun matriks BoW (raw count)
bow_matrix = []
for tokens in tokens_dokumen:
    row = [tokens.count(term) for term in vocab]
    bow_matrix.append(row)

df_bow = pd.DataFrame(bow_matrix, columns=vocab, index=[f'Dok {i+1}' for i in range(len(dokumen))])
print("\nMatriks Bag-of-Words (Raw Count):")
print(df_bow)

Vocabulary: ['beberapa', 'belajar', 'berbagi', 'buatan', 'data', 'dokumen', 'hardware', 'informasi', 'jaringan', 'kecerdasan', 'komputer', 'memungkinkan', 'mencari', 'menggunakan', 'menghubungkan', 'perangkat', 'sistem', 'software', 'temu', 'terdiri']
Jumlah term unik: 20

Matriks Bag-of-Words (Raw Count):
       beberapa  belajar  berbagi  buatan  data  dokumen  hardware  informasi  \
Dok 1         0        0        0       0     0        0         1          0   
Dok 2         1        0        1       0     1        0         0          0   
Dok 3         0        1        0       1     1        0         0          0   
Dok 4         0        0        0       0     0        1         0          1   

       jaringan  kecerdasan  komputer  memungkinkan  mencari  menggunakan  \
Dok 1         0           0         1             0        0            0   
Dok 2         1           0         2             0        0            0   
Dok 3         0           1         1             1    

In [7]:
# Term Frequency (TF) = raw count (sudah sama dengan BoW)
# Tampilkan TF dalam bentuk tabel
df_tf = df_bow.copy()
print("Term Frequency (TF) Manual:")
print(df_tf)

Term Frequency (TF) Manual:
       beberapa  belajar  berbagi  buatan  data  dokumen  hardware  informasi  \
Dok 1         0        0        0       0     0        0         1          0   
Dok 2         1        0        1       0     1        0         0          0   
Dok 3         0        1        0       1     1        0         0          0   
Dok 4         0        0        0       0     0        1         0          1   

       jaringan  kecerdasan  komputer  memungkinkan  mencari  menggunakan  \
Dok 1         0           0         1             0        0            0   
Dok 2         1           0         2             0        0            0   
Dok 3         0           1         1             1        0            0   
Dok 4         0           0         1             0        1            1   

       menghubungkan  perangkat  sistem  software  temu  terdiri  
Dok 1              0          1       1         1     0        1  
Dok 2              1          0       0       

In [8]:
# Document Frequency (df): jumlah dokumen yang mengandung term
N = len(dokumen)
df = {}
for term in vocab:
    df[term] = sum(1 for tokens in tokens_dokumen if term in tokens)

# IDF = log(N / df)
idf = {term: math.log(N / df[term]) for term in vocab}

# Tampilkan tabel df dan IDF
df_idf = pd.DataFrame({
    'Term': vocab,
    'df': [df[t] for t in vocab],
    'IDF = log(N/df)': [round(idf[t], 4) for t in vocab]
})
print(f"Jumlah dokumen (N) = {N}")
print("\nTabel DF dan IDF:")
print(df_idf.to_string(index=False))

Jumlah dokumen (N) = 4

Tabel DF dan IDF:
         Term  df  IDF = log(N/df)
     beberapa   1           1.3863
      belajar   1           1.3863
      berbagi   1           1.3863
       buatan   1           1.3863
         data   2           0.6931
      dokumen   1           1.3863
     hardware   1           1.3863
    informasi   1           1.3863
     jaringan   1           1.3863
   kecerdasan   1           1.3863
     komputer   4           0.0000
 memungkinkan   1           1.3863
      mencari   1           1.3863
  menggunakan   1           1.3863
menghubungkan   1           1.3863
    perangkat   1           1.3863
       sistem   2           0.6931
     software   1           1.3863
         temu   1           1.3863
      terdiri   1           1.3863


In [9]:
# TF-IDF = TF × IDF
tfidf_manual = []
for i, tokens in enumerate(tokens_dokumen):
    row = []
    for term in vocab:
        tf_val = df_tf.loc[f'Dok {i+1}', term]
        idf_val = idf[term]
        row.append(tf_val * idf_val)
    tfidf_manual.append(row)

df_tfidf_manual = pd.DataFrame(tfidf_manual, columns=vocab, index=[f'Dok {i+1}' for i in range(len(dokumen))])
print("Matriks TF-IDF (Manual):")
print(df_tfidf_manual.round(4))

Matriks TF-IDF (Manual):
       beberapa  belajar  berbagi  buatan    data  dokumen  hardware  \
Dok 1    0.0000   0.0000   0.0000  0.0000  0.0000   0.0000    1.3863   
Dok 2    1.3863   0.0000   1.3863  0.0000  0.6931   0.0000    0.0000   
Dok 3    0.0000   1.3863   0.0000  1.3863  0.6931   0.0000    0.0000   
Dok 4    0.0000   0.0000   0.0000  0.0000  0.0000   1.3863    0.0000   

       informasi  jaringan  kecerdasan  komputer  memungkinkan  mencari  \
Dok 1     0.0000    0.0000      0.0000       0.0        0.0000   0.0000   
Dok 2     0.0000    1.3863      0.0000       0.0        0.0000   0.0000   
Dok 3     0.0000    0.0000      1.3863       0.0        1.3863   0.0000   
Dok 4     1.3863    0.0000      0.0000       0.0        0.0000   1.3863   

       menggunakan  menghubungkan  perangkat  sistem  software    temu  \
Dok 1       0.0000         0.0000     1.3863  0.6931    1.3863  0.0000   
Dok 2       0.0000         1.3863     0.0000  0.0000    0.0000  0.0000   
Dok 3       0.00

In [10]:
# Implementasi TF-IDF dengan scikit-learn
# Gunakan teks hasil preprocessing (gabungkan token kembali)
teks_preprocessed = [' '.join(tokens) for tokens in tokens_dokumen]

vectorizer = TfidfVectorizer()
tfidf_sklearn = vectorizer.fit_transform(teks_preprocessed)

# Tampilkan vocabulary dari sklearn
print("Vocabulary Scikit-learn:")
print(vectorizer.get_feature_names_out())

# Tampilkan matriks TF-IDF
df_tfidf_sklearn = pd.DataFrame(
    tfidf_sklearn.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f'Dok {i+1}' for i in range(len(dokumen))]
)
print("\nMatriks TF-IDF (Scikit-learn):")
print(df_tfidf_sklearn.round(4))

Vocabulary Scikit-learn:
['beberapa' 'belajar' 'berbagi' 'buatan' 'data' 'dokumen' 'hardware'
 'informasi' 'jaringan' 'kecerdasan' 'komputer' 'memungkinkan' 'mencari'
 'menggunakan' 'menghubungkan' 'perangkat' 'sistem' 'software' 'temu'
 'terdiri']

Matriks TF-IDF (Scikit-learn):
       beberapa  belajar  berbagi  buatan    data  dokumen  hardware  \
Dok 1    0.0000    0.000   0.0000   0.000  0.0000   0.0000     0.452   
Dok 2    0.4185    0.000   0.4185   0.000  0.3299   0.0000     0.000   
Dok 3    0.0000    0.452   0.0000   0.452  0.3564   0.0000     0.000   
Dok 4    0.0000    0.000   0.0000   0.000  0.0000   0.4119     0.000   

       informasi  jaringan  kecerdasan  komputer  memungkinkan  mencari  \
Dok 1     0.0000    0.0000       0.000    0.2359         0.000   0.0000   
Dok 2     0.0000    0.4185       0.000    0.4367         0.000   0.0000   
Dok 3     0.0000    0.0000       0.452    0.2359         0.452   0.0000   
Dok 4     0.4119    0.0000       0.000    0.2149         0

In [11]:
# Bandingkan bentuk matriks (tidak bisa langsung dibandingkan nilai karena formula berbeda)
print("Perbandingan Dimensi Matriks:")
print(f"Manual     : {df_tfidf_manual.shape}")
print(f"Scikit-learn: {df_tfidf_sklearn.shape}")

print("\nCatatan Perbedaan:")
print("- Manual menggunakan IDF = log(N/df)")
print("- Scikit-learn menggunakan smooth IDF = log((1+N)/(1+df)) + 1")
print("- Scikit-learn melakukan normalisasi L2 pada setiap vektor dokumen")
print("- Scikit-learn juga melakukan preprocessing internal (lowercase, tokenisasi)")

Perbandingan Dimensi Matriks:
Manual     : (4, 20)
Scikit-learn: (4, 20)

Catatan Perbedaan:
- Manual menggunakan IDF = log(N/df)
- Scikit-learn menggunakan smooth IDF = log((1+N)/(1+df)) + 1
- Scikit-learn melakukan normalisasi L2 pada setiap vektor dokumen
- Scikit-learn juga melakukan preprocessing internal (lowercase, tokenisasi)


In [12]:
# Analisis: term dengan bobot TF-IDF tertinggi per dokumen (dari manual)
print("Term dengan Bobot TF-IDF Tertinggi per Dokumen (Manual):\n")
for i in range(len(dokumen)):
    row = df_tfidf_manual.loc[f'Dok {i+1}']
    top_terms = row.sort_values(ascending=False).head(3)
    print(f"Dokumen {i+1}:")
    for term, val in top_terms.items():
        if val > 0:
            print(f"  - {term}: {val:.4f}")
    print()

Term dengan Bobot TF-IDF Tertinggi per Dokumen (Manual):

Dokumen 1:
  - perangkat: 1.3863
  - software: 1.3863
  - terdiri: 1.3863

Dokumen 2:
  - beberapa: 1.3863
  - berbagi: 1.3863
  - menghubungkan: 1.3863

Dokumen 3:
  - belajar: 1.3863
  - buatan: 1.3863
  - kecerdasan: 1.3863

Dokumen 4:
  - mencari: 1.3863
  - menggunakan: 1.3863
  - informasi: 1.3863



In [13]:
# =============================================
# POIN 6: Analisis Term dengan Bobot TF-IDF Tertinggi
# =============================================

# Ambil top-3 term per dokumen dari matriks TF-IDF manual
print("=" * 60)
print("TERM DENGAN BOBOT TF-IDF TERTINGGI PER DOKUMEN")
print("=" * 60)

top_terms_per_doc = {}
for i in range(len(dokumen)):
    row = df_tfidf_manual.loc[f'Dok {i+1}']
    # Urutkan descending, ambil yang > 0
    top = row[row > 0].sort_values(ascending=False).head(3)
    top_terms_per_doc[f'Dok {i+1}'] = top

# Tampilkan dalam DataFrame yang rapi
data_top = []
for doc, top in top_terms_per_doc.items():
    for rank, (term, val) in enumerate(top.items(), 1):
        data_top.append({
            'Dokumen': doc,
            'Peringkat': rank,
            'Term': term,
            'Bobot TF-IDF': round(val, 4)
        })

df_top_terms = pd.DataFrame(data_top)
print(df_top_terms.to_string(index=False))

TERM DENGAN BOBOT TF-IDF TERTINGGI PER DOKUMEN
Dokumen  Peringkat       Term  Bobot TF-IDF
  Dok 1          1   hardware        1.3863
  Dok 1          2  perangkat        1.3863
  Dok 1          3   software        1.3863
  Dok 2          1   beberapa        1.3863
  Dok 2          2    berbagi        1.3863
  Dok 2          3   jaringan        1.3863
  Dok 3          1    belajar        1.3863
  Dok 3          2     buatan        1.3863
  Dok 3          3 kecerdasan        1.3863
  Dok 4          1    dokumen        1.3863
  Dok 4          2  informasi        1.3863
  Dok 4          3    mencari        1.3863


In [14]:
# =============================================
# POIN 6 (Lanjutan): Analisis Mengapa Term Penting
# =============================================

analisis_poin6 = """
ANALISIS TERM DENGAN BOBOT TF-IDF TERTINGGI PER DOKUMEN
=========================================================

1. DOKUMEN 1 — Topik: Perangkat Komputer
   Term tertinggi: "hardware", "perangkat", "software"
   → Mengapa penting?
   Ketiga term ini hanya muncul di Dokumen 1 dan tidak muncul di dokumen lain.
   Karena df = 1, maka IDF-nya maksimal (log(4/1) = 1.3863). Term ini menjadi
   "sidik jari" (fingerprint) dokumen, yang membedakannya dari dokumen lain.
   Jika user mencari "hardware" atau "software", Dokumen 1 akan langsung
   berada di peringkat atas.

2. DOKUMEN 2 — Topik: Jaringan Komputer
   Term tertinggi: "bagi", "beberapa", "menghubungkan"
   → Mengapa penting?
   Term "menghubungkan" sangat spesifik untuk topik jaringan. Meskipun
   "bagi" dan "beberapa" secara semantik kurang kuat, keduanya unik di
   dokumen ini (df = 1) sehingga IDF-nya tinggi. Dalam sistem IR nyata,
   stopword removal yang lebih baik akan membuang "bagi" dan "beberapa",
   menyisakan "menghubungkan" sebagai term kunci.

3. DOKUMEN 3 — Topik: Kecerdasan Buatan
   Term tertinggi: "belajar", "buatan", "kecerdasan"
   → Mengapa penting?
   "kecerdasan" dan "buatan" adalah term paling representatif untuk topik AI.
   Keduanya muncul hanya di Dokumen 3, sehingga IDF maksimal. Term "belajar"
   juga penting karena dalam konteks AI, "belajar" merujuk pada machine learning.

4. DOKUMEN 4 — Topik: Sistem Temu Kembali Informasi
   Term tertinggi: "cari", "dokumen", "informasi"
   → Mengapa penting?
   "temu", "kembali", "informasi" adalah inti dari topik IR. Term "dokumen"
   dan "cari" juga sangat relevan. Semua term ini hanya muncul di Dokumen 4,
   sehingga IDF tinggi dan bobot TF-IDF maksimal.

KESIMPULAN:
Term dengan bobot TF-IDF tertinggi adalah term yang:
(a) Sering muncul di dokumen tertentu (TF tinggi), DAN
(b) Jarang muncul di dokumen lain (IDF tinggi).

Term seperti "komputer" yang muncul di 3 dari 4 dokumen memiliki IDF rendah
(0.2877), sehingga bobot TF-IDF-nya kecil meskipun TF-nya tinggi. Ini
membuktikan bahwa TF-IDF secara otomatis menekan kata umum dan mengangkat
kata spesifik yang benar-benar merepresentasikan topik dokumen. Inilah
mengapa TF-IDF sangat efektif untuk ranking dokumen dalam sistem pencarian.
"""
print(analisis_poin6)


ANALISIS TERM DENGAN BOBOT TF-IDF TERTINGGI PER DOKUMEN

1. DOKUMEN 1 — Topik: Perangkat Komputer
   Term tertinggi: "hardware", "perangkat", "software"
   → Mengapa penting?
   Ketiga term ini hanya muncul di Dokumen 1 dan tidak muncul di dokumen lain.
   Karena df = 1, maka IDF-nya maksimal (log(4/1) = 1.3863). Term ini menjadi
   "sidik jari" (fingerprint) dokumen, yang membedakannya dari dokumen lain.
   Jika user mencari "hardware" atau "software", Dokumen 1 akan langsung
   berada di peringkat atas.

2. DOKUMEN 2 — Topik: Jaringan Komputer
   Term tertinggi: "bagi", "beberapa", "menghubungkan"
   → Mengapa penting?
   Term "menghubungkan" sangat spesifik untuk topik jaringan. Meskipun
   "bagi" dan "beberapa" secara semantik kurang kuat, keduanya unik di
   dokumen ini (df = 1) sehingga IDF-nya tinggi. Dalam sistem IR nyata,
   stopword removal yang lebih baik akan membuang "bagi" dan "beberapa",
   menyisakan "menghubungkan" sebagai term kunci.

3. DOKUMEN 3 — Topik: Kecerdasan

In [15]:
# =============================================
# Analisis singkat
# =============================================

Analisis_singkat = """
Berdasarkan perhitungan TF-IDF secara manual dan menggunakan scikit-learn, dapat
disimpulkan bahwa term dengan bobot tertinggi pada setiap dokumen adalah term yang
bersifat unik dan spesifik (jarang muncul di dokumen lain), seperti "hardware" dan
"software" pada Dokumen 1, "menghubungkan" pada Dokumen 2, "kecerdasan" dan "buatan"
pada Dokumen 3, serta "temu", "kembali", dan "informasi" pada Dokumen 4. Term-term
tersebut penting karena mampu menjadi pembeda utama antar dokumen, sedangkan term umum
seperti "komputer" yang muncul di banyak dokumen justru memiliki bobot rendah karena
kurang diskriminatif. Perbedaan nilai antara perhitungan manual dan scikit-learn
disebabkan oleh perbedaan formula IDF (smooth vs non-smooth) serta normalisasi L2,
namun pola term penting yang dihasilkan tetap konsisten. Dengan demikian, TF-IDF
terbukti efektif dalam menekan kata umum dan mengangkat kata spesifik, sehingga sangat
berguna untuk ranking dokumen dalam sistem temu kembali informasi.
"""
print(Analisis_singkat)


Berdasarkan perhitungan TF-IDF secara manual dan menggunakan scikit-learn, dapat 
disimpulkan bahwa term dengan bobot tertinggi pada setiap dokumen adalah term yang 
bersifat unik dan spesifik (jarang muncul di dokumen lain), seperti "hardware" dan 
"software" pada Dokumen 1, "menghubungkan" pada Dokumen 2, "kecerdasan" dan "buatan" 
pada Dokumen 3, serta "temu", "kembali", dan "informasi" pada Dokumen 4. Term-term 
tersebut penting karena mampu menjadi pembeda utama antar dokumen, sedangkan term umum 
seperti "komputer" yang muncul di banyak dokumen justru memiliki bobot rendah karena 
kurang diskriminatif. Perbedaan nilai antara perhitungan manual dan scikit-learn 
disebabkan oleh perbedaan formula IDF (smooth vs non-smooth) serta normalisasi L2, 
namun pola term penting yang dihasilkan tetap konsisten. Dengan demikian, TF-IDF 
terbukti efektif dalam menekan kata umum dan mengangkat kata spesifik, sehingga sangat 
berguna untuk ranking dokumen dalam sistem temu kembali informasi.

